# ⚡ Decision models, and how to mix models
### When you need an answer and not a text, and how to spend the expensive model only where it counts

**Module 2 · Session 02 · notebook 3 of 3**

An LLM writes its answer word by word, even when all you need is one label. A **decision model** skips the writing. You give
it a text and a typed question, and in one pass through the network it returns a **probability for every allowed answer**.
It cannot chat. Think of it as an `if` statement that understands meaning:

```python
if decide(email, "Is the customer asking for a refund?") > 0.8:
    send_to_refunds_team(email)
```

This kind of model became popular in 2026 with a commercial product called Jev. Open models followed, and several now run in
Ollama. We use `d3:2b`, an open model with 2 billion parameters, which is small enough for everyone to run.

Then comes the main skill of this session: **mixing models**. You rarely want one model for everything. You want the cheap
one for most of the work, and the expensive one only where it counts.

**Before you start:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all.

**Contents**
1. Three kinds of question
2. The same posts, much faster
3. It always answers
4. Does the order of the options matter?
5. Can you trust the confidence?
6. Mix 1: escalate the unsure cases
7. Mix 2: teach a classifier
8. Which tool for which job

**Own machine.** This notebook also runs in Jupyter or VS Code on a laptop with Ollama installed: Moodle explains how. Leave `BACKEND` on `ollama`.

**Plan B, if local does not work** (no free GPU, or Ollama fails). Create a free key at [aistudio.google.com](https://aistudio.google.com) and
store it as a Colab secret called `GEMINI_API_KEY` (key icon on the left, then switch on access for this notebook). In the setup cell, set `BACKEND`
to `gemini` and run everything again. A Groq key from [console.groq.com](https://console.groq.com), stored as `GROQ_API_KEY`, works the same way with `groq`.

The code stays the same and your texts go to the provider. Free keys allow few calls a minute and about 500 a day, so everyone needs their own key, and plan B runs on smaller samples and takes longer. Your numbers will differ from the slides, because the cloud model is stronger.

The cloud has no decision model, so here plan B replays saved answers from a real `d3:2b` run. Those parts run instantly, and new questions to the decision model do not work.

In [ ]:
# @title Setup: install Ollama and download the models (2 to 4 minutes) { display-mode: "form" }
BACKEND = "ollama"  # @param ["ollama", "gemini", "groq"]
%pip install -q -U openai sentence-transformers 2> /dev/null
import hashlib, json, os, random, shutil, subprocess, sys, time, warnings
from concurrent.futures import ThreadPoolExecutor
from typing import Literal, Optional
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

OLLAMA = "http://localhost:11434"
DECIDER = "d3:2b"
DEVICE = "cpu" if sys.platform == "darwin" else None   # on a Mac, Ollama already fills the GPU memory
if BACKEND == "ollama":
    if shutil.which("ollama") is None and "google.colab" in sys.modules:
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama is not installed. On your own machine, install it as described on Moodle. Otherwise use plan B: see the top of this notebook.")

    # start_new_session keeps the server alive when you stop a cell
    subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    for model in ["d3:2b", "gemma4:e2b"]:
        t0 = time.time()
        done = subprocess.run(["ollama", "pull", model], capture_output=True)
        print(f"{model} ready ({time.time() - t0:.0f} s)" if done.returncode == 0 else f"{model} did not download. Use plan B: see the top of this notebook.")

    if "google.colab" in sys.modules:
        gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
        print(gpu if gpu else "No GPU on this runtime. Each text would take 10 to 20 seconds, which is too slow here. Use Runtime → Change runtime type → T4 GPU. If Colab has none free, use plan B: see the top of this notebook.")
    BASE_URL, API_KEY, MODEL, EXTRA, PAUSE = OLLAMA + "/v1", "ollama", "gemma4:e2b", {"reasoning_effort": "none"}, 0
else:   # plan B: the same calls go to a cloud model. Free keys allow 15 (Gemini) or 30 (Groq) calls a minute
    BASE_URL, secret, MODEL, EXTRA, PAUSE = {
        "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "GEMINI_API_KEY", "gemini-flash-lite-latest", {}, 4.2),
        "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-20b", {"reasoning_effort": "low"}, 3),
    }[BACKEND]
    try:
        from google.colab import userdata
        API_KEY = userdata.get(secret)
    except ImportError:
        API_KEY = os.environ[secret]

    # the cloud has no decision model, so calls to it are answered from a saved d3:2b run
    SAVED = requests.get("https://raw.githubusercontent.com/aaubs/ds-master/main/data/llm_session/decide_nb3.json").json()
    class Replay:
        def __init__(self, payload):
            key = hashlib.md5(json.dumps([payload["state"], payload["questions"]]).encode()).hexdigest()
            if key not in SAVED:
                raise ValueError("Plan B only has the saved answers. A new question needs a decision model, so the Ollama route.")
            self.answers = SAVED[key]
        def json(self):
            return {"answers": self.answers}
    requests.post = lambda url, json, timeout: Replay(json)

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
pd.set_option("display.max_colwidth", 200)

---
# ❓ 1 · Three kinds of question

We talk to the decision model with a plain web request. The request holds the model name, the text (called `state` here) and your question. It has no `system`
or `user` message, and no prompt in the usual sense. The first call can take up to three minutes while the model loads.

In [ ]:
POST = "My landlord raised the rent by 20 percent and I can't sleep from the worry. Can he do that? What are my options?"

payload = {
    "model": DECIDER,
    "state": POST,
    "questions": {"asks_for_advice": {"type": "noul", "instructions": "Does the writer ask for advice?"}},
}

response = requests.post(OLLAMA + "/v1/systemone", json=payload, timeout=180)
print(json.dumps(response.json(), indent=2)[:500])

You get a number and not a sentence. There are three types of question:

| Type | What you get | In code it works like |
|---|---|---|
| `noul` (this API's word for yes or no) | the probability that the answer is yes | `if` |
| `choice` | a probability for each option | `switch` |
| `score` | a position on a scale you define | a number to sort by |

You can ask several questions about one text in a single call.

In [ ]:
QUESTIONS = {
    "asks_for_advice": {"type": "noul", "instructions": "Does the writer ask for advice?"},
    "topic": {"type": "choice", "instructions": "What is this post mainly about?",
              "criteria": {"money": "Money, rent, debt", "love": "Partners and dating", "work": "Jobs and studies"}},
    "distress": {"type": "score", "instructions": "How distressed does the writer sound?",
                 "criteria": ["Calm", "A bit bothered", "Upset", "Desperate"]},
}

payload = {"model": DECIDER, "state": POST, "questions": QUESTIONS}
response = requests.post(OLLAMA + "/v1/systemone", json=payload, timeout=180)
answers = response.json()["answers"]

print(answers["asks_for_advice"])
print(answers["topic"])
print(answers["distress"])

Only the text and the questions change from call to call, so this becomes a short function.

In [ ]:
def decide(state, questions):
    payload = {"model": DECIDER, "state": state, "questions": questions}
    response = requests.post(OLLAMA + "/v1/systemone", json=payload, timeout=180)
    return response.json()["answers"]

---
# 🏎️ 2 · The same posts, much faster

The same 115 test posts and the same one-line definitions as notebook 1. Friday's trained classifiers reached 85.2 % and 88.7 %.

In [ ]:
df = pd.read_json("https://raw.githubusercontent.com/aaubs/ds-master/main/data/dk_posts_synth_en_processed.json")
df["text"] = df["title"] + ". " + df["selftext"]
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["label"])
if BACKEND != "ollama":
    test = test.head(40)   # plan B: free keys allow few calls a minute

CODEBOOK = {
    "Love & Dating": "Romantic relationships, dating, partners, jealousy, breakups.",
    "Family Dynamics": "Siblings, parents, children, in-laws, family boundaries.",
    "Work, Study & Career": "Jobs, colleagues, bosses, studies, career choices.",
    "Friendship & Social Life": "Friends, social circles, loneliness, fitting in.",
    "Health & Wellness (Physical and Mental)": "Physical or mental health, stress, sleep, diet.",
    "Personal Finance & Housing": "Money, debt, rent, buying a home.",
    "Practical Questions & Everyday Life": "How-to questions, household tasks, small practical things.",
    "Everyday Observations & Rants": "Complaints or remarks about daily life, no real question.",
}

The codebook goes straight into the question as the `criteria`. We try it on one post first.

In [ ]:
QUESTION = {"category": {"type": "choice",
                         "instructions": "Which category of an advice column fits this post best?",
                         "criteria": CODEBOOK}}

answer = decide(test["text"].iloc[0], QUESTION)["category"]
print(answer)

Now all the posts. We keep the answer and the confidence of each one.

In [ ]:
t0 = time.time()
dec_pred = []
dec_conf = []
for text in test["text"]:
    answer = decide(text, QUESTION)["category"]
    dec_pred.append(answer["choice"])
    dec_conf.append(answer["confidence"])
dec_seconds = (time.time() - t0) / len(test)
if BACKEND != "ollama":
    dec_seconds = 0.11   # replayed, so we use the time measured on a T4

dec_conf = np.array(dec_conf)
print(f"decision model: {accuracy_score(test['label'], dec_pred):.1%} · {dec_seconds:.2f} s per post")

Now the small LLM on the same posts, with the `ask()` function from notebook 1.

In [ ]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(BASE_URL, "·", MODEL)



def ask(system, user, schema):
    for temperature in (0, 0.2, 0.4):   # the same temperature would repeat the same mistake
        time.sleep(PAUSE)
        try:
            response = client.chat.completions.parse(
                model=MODEL,
                temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                response_format=schema,
                **EXTRA,
            )
            return response.choices[0].message.parsed
        except Exception as e:
            error = e
            if "429" in str(e):
                time.sleep(30)   # over the free limit: wait for the next minute
    raise error

In [ ]:
class Post(BaseModel):
    category: Literal[tuple(CODEBOOK)]

SYSTEM = "Classify this post to an advice column. Categories:\n"
for label, definition in CODEBOOK.items():
    SYSTEM += f"- {label}: {definition}\n"

ask(SYSTEM, "warming up the model", Post)

t0 = time.time()
llm_pred = []
for text in test["text"]:
    llm_pred.append(ask(SYSTEM, text, Post).category)
llm_seconds = (time.time() - t0) / len(test)
if BACKEND != "ollama":
    llm_seconds = 0.31   # measured on a T4: the pause for the free limit would distort the comparison

print(f"small LLM: {accuracy_score(test['label'], llm_pred):.1%} · {llm_seconds:.2f} s per post")
print(f"The decision model takes {dec_seconds / llm_seconds:.0%} of the LLM's time on this machine.")

---
# 🗑️ 3 · It always answers

A decision model has to pick from your options. What happens when no option fits?

In [ ]:
NONSENSE = ["asdf qwerty 123", "The mitochondria is the powerhouse of the cell."]

for text in NONSENSE:
    answer = decide(text, QUESTION)["category"]
    print(f"{text!r:50} → {answer['choice']} (confidence {answer['confidence']:.2f})")

Now we give it a way out: one more option.

In [ ]:
WITH_EXIT = {"category": {"type": "choice",
                          "instructions": "Which category of an advice column fits this post best?",
                          "criteria": {**CODEBOOK, "None of these": "The text does not fit any category, or is not a post at all."}}}

for text in NONSENSE:
    answer = decide(text, WITH_EXIT)["category"]
    print(f"{text!r:50} → {answer['choice']} (confidence {answer['confidence']:.2f})")

🤔 *High confidence means the model prefers this option over the others. It does not mean the answer makes sense. When
you design options for real data, always ask what happens to the text that fits none of them.*

---
# 🔀 4 · Does the order of the options matter?

It should not. We shuffle the options and count how many answers change. It is a cheap test of how stable a model is, and it
costs one extra pass.

In [ ]:
items = list(CODEBOOK.items())
random.Random(3).shuffle(items)
print([label for label, definition in items][:3], "...")

SHUFFLED = {"category": {"type": "choice",
                         "instructions": "Which category of an advice column fits this post best?",
                         "criteria": dict(items)}}

shuffled = []
for text in test["text"]:
    shuffled.append(decide(text, SHUFFLED)["category"]["choice"])

changed = np.mean(np.array(dec_pred) != np.array(shuffled))
print(f"{changed:.1%} of answers changed when only the order changed")

---
# 🎚️ 5 · Can you trust the confidence?

Before we use a confidence number to make decisions, we check it. Do high-confidence answers really turn out right more often?

First the LLM, asked to rate its own confidence. Then the decision model, whose number comes directly from its probabilities.

In [ ]:
class PostConf(BaseModel):
    category: Literal[tuple(CODEBOOK)]
    confidence: float = Field(description="Your confidence from 0 to 1")

own = []
for text in test["text"]:
    own.append(ask(SYSTEM + "\nAlso give your confidence from 0 to 1.", text, PostConf))

llm_check = pd.DataFrame({"model": "LLM's own rating",
                          "conf": [o.confidence for o in own],
                          "right": [o.category for o in own] == test["label"].values})
dec_check = pd.DataFrame({"model": "decision model",
                          "conf": dec_conf,
                          "right": np.array(dec_pred) == test["label"].values})

checks = pd.concat([llm_check, dec_check])
checks["band"] = pd.cut(checks["conf"], [-.01, .5, .8, .95, 1.01], labels=["up to 0.5", "0.5–0.8", "0.8–0.95", "above 0.95"])
checks.groupby(["model", "band"], observed=True)["right"].agg(accuracy="mean", posts="count").round(2)

🤔 *Which number can you act on? Asking a model how sure it is gives you one more piece of generated text. A probability
from the model's output layer is a measurement, although it can still be off, so you check it against labels as we just did.*

That check gives you **three bands**. Above a high threshold you **accept** the answer. In the middle, a second model checks it.
At the bottom, a person looks at it.

---
# 🪜 6 · Mix 1: escalate the unsure cases

The decision model takes every post. Only the posts below a confidence threshold go on to the LLM. How much quality do we
get back, and what does it cost in time?

In [ ]:
llm_right = np.array(llm_pred) == test["label"].values
dec_right = np.array(dec_pred) == test["label"].values
rows = []
for threshold in [0, .5, .7, .8, .9, .95, 1.01]:
    unsure = dec_conf < threshold
    right = np.where(unsure, llm_right, dec_right)
    seconds = dec_seconds + unsure.mean() * llm_seconds
    rows.append({"threshold": threshold, "sent to LLM": unsure.mean(), "accuracy": right.mean(), "seconds per post": seconds})
mix = pd.DataFrame(rows)
mix.style.format({"threshold": "{:.2f}", "sent to LLM": "{:.0%}", "accuracy": "{:.1%}", "seconds per post": "{:.2f}"})

In [ ]:
# @title Chart: quality against time { display-mode: "form" }
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(mix["seconds per post"], mix["accuracy"] * 100, "-o", color=NAVY)
for r in mix.itertuples():
    ax.annotate(f"{r[2]:.0%} sent on", (r[4], r.accuracy * 100), textcoords="offset points", xytext=(4, -12), fontsize=8, color=GREY)
ax.set_xlabel("seconds per post"); ax.set_ylabel("accuracy, %")
plt.show()

🤔 *Find the threshold where you get most of the quality for a small part of the time. What would you pick for a million posts?
What if a wrong answer costs a lot?*

---
# 🎓 7 · Mix 2: teach a classifier

An even cheaper route at volume. Let a model label a sample, then train a classifier on those labels, as you did on Friday. The classifier
needs no language model at all, and it handles hundreds of texts per second. Nobody labelled anything by hand.

The decision model labels the 342 training posts. We train the best model from Friday, sentence embeddings with a linear SVM, and test on the real labels.
The labels are not perfect, so we try two versions: all of them, and only those where the decision model was confident.

In [ ]:
t0 = time.time()
labels = []
confidence = []
for text in train["text"]:
    answer = decide(text, QUESTION)["category"]
    labels.append(answer["choice"])
    confidence.append(answer["confidence"])
teach_seconds = time.time() - t0

labels = np.array(labels)
confident = np.array(confidence) >= 0.8
print(f"labelling {len(train)} posts took {teach_seconds:.0f} s")
print(f"{(labels == train['label'].values).mean():.0%} of the labels are right · {confident.mean():.0%} are confident")

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
X_train = embedder.encode(train["text"].tolist(), show_progress_bar=False)
t0 = time.time()
X_test = embedder.encode(test["text"].tolist(), show_progress_bar=False)

student_all = LinearSVC().fit(X_train, labels)
print(f"classifier on all labels:              {accuracy_score(test['label'], student_all.predict(X_test)):.1%}")

student = LinearSVC().fit(X_train[confident], labels[confident])
print(f"classifier on confident labels only:   {accuracy_score(test['label'], student.predict(X_test)):.1%}")
student_seconds = (time.time() - t0) / len(test)

print(f"the decision model itself:             {accuracy_score(test['label'], dec_pred):.1%}")
print("Friday, trained on human labels:       88.7 %")

🤔 *The classifier learns the mistakes of its teacher as well as its skills. Why does it help to keep only the confident labels? What would you check before
you trust it? And what can a classifier not do that an LLM can?* (Think of notebook 2: it can learn a label, but it cannot fill in a form.)

---
# 🧰 8 · Which tool for which job

In [ ]:
N = 1_000_000
speeds = {"trained classifier": student_seconds, "decision model": dec_seconds, "small LLM": llm_seconds}
print(f"Time for {N:,} short texts on this runtime:")
for name, s in speeds.items():
    print(f"  {name:<20} {s * N / 3600:>8.2f} hours")

| Tool | Use it when | Weak spot |
|---|---|---|
| **Rules and keywords** | the pattern is exact, such as a date or an order number | breaks on new wording |
| **Trained classifier** (TF-IDF, embeddings) | stable labels, you have labels, huge volume | needs labels, only gives labels |
| **Topic model** (BERTopic) | you do not know the categories yet | groups need a human to name and judge them |
| **Decision model** | no labels yet, categories change, many small decisions, speed matters | no explanation, cannot write, always picks something |
| **Small LLM** | you need text out: extract, rewrite, summarise, on a few hundred to a few thousand items | slower, and weak on subtle judgement |
| **Big cloud LLM** | the hardest cases, the best quality, small volume | cost, limits, text leaves your hands |

**The habit to take away:** use the LLM on a sample to find structure and make labels, and use cheaper models for the volume.
Send only the unsure cases back up. Whatever you build, compare it with a simple baseline and read the mistakes.

The assignment uses all of this on one question.